# IoT Anomaly Detection: Data download
CIC IoT-DIAD 2024 · Ledja Halltari · BCIT COMP 9080

This notebook downloads the CIC IoT-DIAD 2024 dataset, converts it from CSV to Parquet, and saves it to Google Drive. **It only needs to run once.** The analysis is in the separate notebook `IoT_Anomaly_Detection_clean`.

**Steps:**
1. Settings
2. Downloader functions
3. Register on the CIC website (fill in your details first)
4. List all files
5. Download everything to Colab's disk
6. Check the download
7. Convert CSV → Parquet
8. Copy the Parquet files to Drive
9. Verify the copy

**Result:** 312 Parquet files + 2 README files (11.29 GB) in `MyDrive/CIC-IoT-DIAD-2024-parquet`. The original CSVs are 54.5 GB. Colab's disk is temporary, so steps 5–9 must run in the same session.

## 1. Settings

In [2]:
# ========== SETTINGS ==========
# CSVs (about 54.5 GB) are downloaded to Colab's own disk, converted to Parquet (about 11.3 GB),
# and only the Parquet files are copied to Drive.
import os, shutil, time
from google.colab import drive
drive.mount("/content/drive")

START_URL    = "https://cicresearch.ca/IOTDataset/CIC-IoT-IDAD-Dataset-2024/browse.php"  # updated by the registration cell
ONLY_FOLDERS = []      # [] = everything

PROJECT   = "/content/drive/MyDrive/IoT_Anomaly_Detection_Project/Data"
OUT_DIR   = "/content/data"                          # CSVs (Colab disk, temporary)
PQ_LOCAL  = "/content/parquet"                       # Parquet (Colab disk, temporary)
PQ_DRIVE  = f"{PROJECT}/CIC-IoT-DIAD-2024-parquet"   # Parquet (Drive, permanent)

os.makedirs(OUT_DIR, exist_ok=True)
print("Project folder found:", os.path.isdir(PROJECT))
print("Old dataset folder still there?", os.path.isdir(PQ_DRIVE), "(should be False after deleting)")
print(f"Colab disk free: {shutil.disk_usage('/content').free/1024**3:.1f} GB  (need about 70 GB)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder found: True
Old dataset folder still there? False (should be False after deleting)
Colab disk free: 193.2 GB  (need about 70 GB)


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Downloader functions
Crawls the CIC folder pages, finds every file, and downloads with resume support (broken downloads continue where they stopped).

In [3]:
import os, re, time, html
from urllib.parse import urljoin, urlparse, parse_qs, urlencode, unquote
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

FILE_EXTS = (".csv", ".zip", ".txt", ".pcap", ".pcapng", ".gz", ".tar", ".7z", ".rar",
             ".parquet", ".json", ".xlsx", ".pdf", ".md")
SKIP_TEXT = {"..", "parent directory", "parent", "back", "up", "home", "../"}

def make_session(cookie_string=""):
    s = requests.Session()
    s.headers.update({
        "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                       "(KHTML, like Gecko) Chrome/126.0 Safari/537.36"),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    })
    for part in cookie_string.split(";"):
        if "=" in part:
            k, v = part.strip().split("=", 1)
            s.cookies.set(k, v)
    return s

def clean_name(text):
    text = html.unescape(text)
    text = "".join(ch for ch in text if ch.isprintable() and ord(ch) < 0x2000 or ch.isalnum())
    text = re.sub(r'[\\/:*?"<>|]+', "_", text).strip(" ._")
    return text or "unnamed"

def _key(url):
    p = urlparse(url)
    q = {k: v for k, v in parse_qs(p.query).items() if k.lower() != "t"}
    return p._replace(query=urlencode(sorted(q.items()), doseq=True), fragment="").geturl()

def classify(url, text, base_netloc):
    p = urlparse(url)
    if p.scheme not in ("http", "https") or (p.netloc and p.netloc != base_netloc):
        return None
    if text.strip().lower() in SKIP_TEXT:
        return None
    last = unquote(p.path.rsplit("/", 1)[-1]).lower()
    blob = (last + " " + unquote(p.query) + " " + text).lower()
    if any(blob.rstrip().endswith(e) or (e + " ") in blob or (e + "&") in blob for e in FILE_EXTS) \
            or any(e in unquote(p.query).lower() for e in FILE_EXTS):
        return "file"
    if "download" in (last + p.query).lower() or "dl.php" in last or "get.php" in last:
        return "file"
    if "browse" in last or p.path.endswith("/"):
        return "folder"
    return None

def get_links(session, url):
    r = session.get(url, timeout=60)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"].strip()
        if href.startswith(("#", "mailto:", "javascript:")) or href.startswith("?C="):
            continue
        links.append((urljoin(r.url, href), a.get_text(" ", strip=True)))
    return r, soup, links

def crawl(session, start_url, only_folders=None, verbose=True):
    base_netloc = urlparse(start_url).netloc
    seen = {_key(start_url)}
    files, queue = [], [(start_url, "")]
    while queue:
        url, rel = queue.pop(0)
        if verbose:
            print("📂 scanning:", rel or "/")
        _, _, links = get_links(session, url)
        for link, text in links:
            kind = classify(link, text, base_netloc)
            k = _key(link)
            if kind == "folder":
                if k in seen:
                    continue
                seen.add(k)
                name = clean_name(text)
                sub = f"{rel}/{name}" if rel else name
                if only_folders and not rel and not any(f.lower() in name.lower() for f in only_folders):
                    if verbose: print("   skipping folder:", name)
                    continue
                queue.append((link, sub))
            elif kind == "file" and k not in seen:
                seen.add(k)
                name = clean_name(text)
                if "." not in name:
                    p = urlparse(link)
                    cands = [unquote(v) for vs in parse_qs(p.query).values() for v in vs] + [unquote(p.path)]
                    cands = [c.rsplit("/", 1)[-1] for c in cands if "." in c.rsplit("/", 1)[-1]]
                    name = clean_name(cands[0]) if cands else name
                files.append({"url": link, "path": f"{rel}/{name}" if rel else name})
    return files

def human(n):
    if n is None: return "?"
    for u in ["B", "KB", "MB", "GB", "TB"]:
        if n < 1024: return f"{n:.1f} {u}"
        n /= 1024
    return f"{n:.1f} PB"

def remote_size(session, url):
    try:
        r = session.head(url, allow_redirects=True, timeout=30)
        n = r.headers.get("Content-Length")
        if n and "text/html" not in r.headers.get("Content-Type", ""):
            return int(n)
    except Exception:
        pass
    return None

def download_file(session, url, dest, retries=5, chunk=4 * 1024 * 1024):
    os.makedirs(os.path.dirname(dest) or ".", exist_ok=True)
    part = dest + ".part"
    for attempt in range(1, retries + 1):
        try:
            done = os.path.getsize(part) if os.path.exists(part) else 0
            headers = {"Range": f"bytes={done}-"} if done else {}
            with session.get(url, stream=True, timeout=120, headers=headers) as r:
                if r.status_code == 416:
                    break
                r.raise_for_status()
                if "text/html" in r.headers.get("Content-Type", "") and not dest.lower().endswith((".html", ".htm")):
                    raise RuntimeError("Server sent a web page instead of the file. Your access has probably "
                                       "expired: get a fresh link/cookie from the browser and re-run.")
                if done and r.status_code != 206:
                    done = 0
                total = r.headers.get("Content-Length")
                total = int(total) + done if total else None
                with open(part, "ab" if done else "wb") as f, tqdm(total=total, initial=done, unit="B",
                        unit_scale=True, unit_divisor=1024, desc=os.path.basename(dest)[:40], leave=False) as bar:
                    for data in r.iter_content(chunk_size=chunk):
                        f.write(data)
                        bar.update(len(data))
            os.replace(part, dest)
            return os.path.getsize(dest)
        except RuntimeError:
            raise
        except Exception as e:
            wait = min(60, 5 * attempt)
            print(f"   ⚠️ {os.path.basename(dest)}: {e} — retry {attempt}/{retries} in {wait}s")
            time.sleep(wait)
    if os.path.exists(part):
        os.replace(part, dest)
        return os.path.getsize(dest)
    raise RuntimeError(f"Failed after {retries} attempts: {url}")

def download_all(session, files, out_dir, skip_existing=True):
    ok, failed = 0, []
    for i, f in enumerate(files, 1):
        dest = os.path.join(out_dir, f["path"])
        if skip_existing and os.path.exists(dest) and os.path.getsize(dest) > 0:
            exp = f.get("size")
            if exp is None or os.path.getsize(dest) == exp:
                print(f"[{i}/{len(files)}] ✔ already have {f['path']}")
                ok += 1
                continue
        print(f"[{i}/{len(files)}] ⬇ {f['path']}  ({human(f.get('size'))})")
        try:
            download_file(session, f["url"], dest)
            ok += 1
        except RuntimeError as e:
            print("   ❌", e)
            failed.append(f)
            if "expired" in str(e):
                print("Stopping — refresh access and run again (finished files are kept).")
                break
    print(f"\nDone: {ok} ok, {len(failed)} failed.")
    return failed

print("Downloader ready ✅")

Downloader ready ✅


## 3. Register on the CIC website
The download link only works for a registered session. This cell fills in the registration form automatically. **Replace the placeholder details with your own before running.**

In [4]:
REG_URL = "https://cicresearch.ca/IOTDataset/CIC-IoT-IDAD-Dataset-2024/"

# Your details for the CIC registration form: fill these in before running.
MY_INFO = {
    "first":   "FIRST_NAME",
    "last":    "LAST_NAME",
    "name":    "FULL NAME",
    "email":   "your.email@example.com",
    "org":     "YOUR_INSTITUTION",
    "company": "YOUR_INSTITUTION",
    "institution": "YOUR_INSTITUTION",
    "country": "Canada",
    "purpose": "Academic research on IoT anomaly detection",
    "job":     "Student",
    "position": "Student",
}

session = make_session("")
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/153.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
})

r = session.get(REG_URL, timeout=60)
soup = BeautifulSoup(r.text, "html.parser")
form = soup.find("form")
if form is None:
    print("No form found. Status:", r.status_code); print(r.text[:2000])
else:
    labels = {l.get("for"): l.get_text(" ", strip=True) for l in soup.find_all("label") if l.get("for")}

    def guess(el, name):
        hint = " ".join([name, el.get("id") or "", el.get("placeholder") or "",
                         labels.get(el.get("id"), "")]).lower()
        for key in sorted(MY_INFO, key=len, reverse=True):
            if key in hint:
                return MY_INFO[key]
        if "mail" in hint:
            return MY_INFO["email"]
        return "N/A"

    data = {}
    print("Filling in the form:")
    for el in form.find_all(["input", "select", "textarea"]):
        name = el.get("name")
        if not name:
            continue
        typ = (el.get("type") or el.name).lower()
        if typ in ("submit", "button", "image", "reset"):
            if el.get("value") is not None:
                data.setdefault(name, el.get("value"))
            continue
        if typ == "hidden":
            val = el.get("value", "")
        elif typ in ("checkbox", "radio"):
            if name in data:
                continue
            val = el.get("value", "on")
        elif typ == "select":
            opts = [(o.get("value") if o.get("value") is not None else o.get_text(strip=True), o.get_text(strip=True))
                    for o in el.find_all("option")]
            opts = [o for o in opts if o[0]]
            want = guess(el, name).lower()
            match = [v for v, t in opts if want in t.lower() or want == v.lower()]
            val = match[0] if match else (opts[0][0] if opts else "")
        else:
            val = guess(el, name)
        data[name] = val
        if typ != "hidden":
            print(f"  {name:<20} = {val}")

    action = urljoin(r.url, form.get("action") or r.url)
    method = (form.get("method") or "get").lower()
    session.headers["Referer"] = r.url
    if method == "post":
        r2 = session.post(action, data=data, timeout=60, allow_redirects=True)
    else:
        r2 = session.get(action, params=data, timeout=60, allow_redirects=True)

    print("\nSubmitted. Status:", r2.status_code, "| ended at:", r2.url)
    print("Cookies received:", session.cookies.get_dict())

    START_URL = r2.url if "browse" in r2.url else urljoin(REG_URL, f"browse.php?t={int(time.time()*1000)}")
    resp, soup2, links = get_links(session, START_URL)
    base = urlparse(START_URL).netloc
    useful = [(classify(u, t, base), t) for u, t in links if classify(u, t, base)]
    if useful:
        print("\n✅ Connected! START_URL =", START_URL)
        for kind, text in useful:
            print(f"  {'📁' if kind=='folder' else '📄'} {text}")
    else:
        print("\n❌ Still no folders. Page says:"); print(resp.text[:1500])

Filling in the form:
  first_name           = FIRST_NAME
  last_name            = LAST_NAME
  email                = your.email@example.com
  institution          = YOUR_INSTITUTION
  job_title            = Student
  country              = Canada

Submitted. Status: 200 | ended at: https://cicresearch.ca/IOTDataset/CIC-IoT-IDAD-Dataset-2024/insert.php
Cookies received: {'Token': 'gifadbm99klh129qnf891ar5p6'}

✅ Connected! START_URL = https://cicresearch.ca/IOTDataset/CIC-IoT-IDAD-Dataset-2024/browse.php?t=1791082832060
  📁 Anomaly Detection - Flow Based features
  📁 Device Identification_Anomaly Detection - Packet Based Features


## 4. List all files
Scans every folder on the site. Expected: 314 files (312 CSVs + 2 README files).

In [5]:
import pandas as pd
files = crawl(session, START_URL, only_folders=ONLY_FOLDERS)
print(f"\nChecking sizes of {len(files)} files...")
for f in tqdm(files):
    f["size"] = remote_size(session, f["url"])

pd.set_option("display.max_rows", 500, "display.max_colwidth", 120)
display(pd.DataFrame([{"path": f["path"], "size": human(f["size"])} for f in files]))

total = sum(f["size"] or 0 for f in files)
free = shutil.disk_usage(OUT_DIR).free
print(f"\nTotal: {len(files)} files, {human(total)} | Free space: {human(free)}")
if total > free * 0.95:
    print("⚠️ NOT ENOUGH SPACE. Use ONLY_FOLDERS in Cell 1 to download less.")

📂 scanning: /
📂 scanning: Anomaly Detection - Flow Based features
📂 scanning: Device Identification_Anomaly Detection - Packet Based Features
📂 scanning: Anomaly Detection - Flow Based features/Benign
📂 scanning: Anomaly Detection - Flow Based features/BruteForce
📂 scanning: Anomaly Detection - Flow Based features/DDoS
📂 scanning: Anomaly Detection - Flow Based features/DoS
📂 scanning: Anomaly Detection - Flow Based features/Mirai
📂 scanning: Anomaly Detection - Flow Based features/Recon
📂 scanning: Anomaly Detection - Flow Based features/Spoofing
📂 scanning: Anomaly Detection - Flow Based features/Web-Based
📂 scanning: Device Identification_Anomaly Detection - Packet Based Features/BenignTraffic
📂 scanning: Device Identification_Anomaly Detection - Packet Based Features/BruteForce
📂 scanning: Device Identification_Anomaly Detection - Packet Based Features/DDoS
📂 scanning: Device Identification_Anomaly Detection - Packet Based Features/DoS
📂 scanning: Device Identification_Anomaly Dete

  0%|          | 0/314 [00:00<?, ?it/s]

,path,size
0,Anomaly Detection - Flow Based features/README.txt,?
1,Device Identification_Anomaly Detection - Packet Based Features/README.txt,?
2,Anomaly Detection - Flow Based features/Benign/BenignTraffic.pcap_Flow.csv,?
3,Anomaly Detection - Flow Based features/Benign/BenignTraffic1.pcap_Flow.csv,?
4,Anomaly Detection - Flow Based features/Benign/BenignTraffic2.pcap_Flow.csv,?
5,Anomaly Detection - Flow Based features/Benign/BenignTraffic3.pcap_Flow.csv,?
6,Anomaly Detection - Flow Based features/BruteForce/DictionaryBruteForce.pcap_Flow.csv,?
7,Anomaly Detection - Flow Based features/Mirai/Mirai-greeth_flood.pcap_Flow.csv,?
8,Anomaly Detection - Flow Based features/Mirai/Mirai-greeth_flood1.pcap_Flow.csv,?
9,Anomaly Detection - Flow Based features/Mirai/Mirai-greeth_flood10.pcap_Flow.csv,?



Total: 314 files, 0.0 B | Free space: 193.2 GB


## 5. Download everything
Downloads to Colab's disk. Already-downloaded files are skipped, so you can re-run this cell if it stops.

In [6]:
# 1) Test with one small file (the README)
readme = next(f for f in files if f["path"].endswith("README.txt"))
dest = os.path.join(OUT_DIR, readme["path"])
download_file(session, readme["url"], dest)
print("✅ Test download worked! README starts with:\n")
print(open(dest, errors="ignore").read()[:500])

# 2) Download everything (files already on disk are skipped, so you can re-run after a break)
print("\nStarting full download of", len(files), "files...\n")
failed = []
for i, f in enumerate(files, 1):
    free_gb = shutil.disk_usage(OUT_DIR).free / 1024**3
    if free_gb < 2:
        print(f"⚠️ Stopped: only {free_gb:.1f} GB free left. Downloaded {i-1} of {len(files)} files.")
        break
    d = os.path.join(OUT_DIR, f["path"])
    if os.path.exists(d) and os.path.getsize(d) > 0:
        continue
    print(f"[{i}/{len(files)}] {f['path']}")
    try:
        download_file(session, f["url"], d)
    except Exception as e:
        print("   ❌", e)
        failed.append(f)
print(f"\nDone. {len(failed)} failed.")

README.txt: 0.00B [00:00, ?B/s]

✅ Test download worked! README starts with:

The CICFlowMeter is an open source tool that generates Biflows from pcap files, and extracts features from these flows.

NOTE: TCP flows are usually terminated upon connection teardown (by FIN packet) while UDP flows are terminated by a flow timeout. The flow timeout value can be assigned arbitrarily by the individual scheme e.g., 600 seconds for both TCP and UDP.

NOTE:

The data found in this folder is comprised of 8 folders(Benign + 7 Attacks). Each folder will contain csv files for the kind 

Starting full download of 314 files...

[2/314] Device Identification_Anomaly Detection - Packet Based Features/README.txt


README.txt: 0.00B [00:00, ?B/s]

[3/314] Anomaly Detection - Flow Based features/Benign/BenignTraffic.pcap_Flow.csv


BenignTraffic.pcap_Flow.csv: 0.00B [00:00, ?B/s]

...

[313/314] Device Identification_Anomaly Detection - Packet Based Features/Web-Based/Uploading_Attack/Uploading_Attack.csv


Uploading_Attack.csv: 0.00B [00:00, ?B/s]

[314/314] Device Identification_Anomaly Detection - Packet Based Features/Web-Based/XSS/XSS.csv


XSS.csv: 0.00B [00:00, ?B/s]


Done. 0 failed.


## 6. Check the download
Looks for missing, unfinished, empty, or broken files and re-downloads anything missing.

In [7]:
import glob
all_files = [p for p in glob.glob(f"{OUT_DIR}/**/*", recursive=True) if os.path.isfile(p)]
csvs      = [p for p in all_files if p.endswith(".csv")]
partial   = [p for p in all_files if p.endswith(".part")]
empty     = [p for p in all_files if os.path.getsize(p) == 0]
html_junk = []
for p in csvs:
    with open(p, "rb") as f:
        if f.read(200).lstrip().lower().startswith((b"<", b"<!doctype")):
            html_junk.append(p)
missing = [f for f in files if not os.path.exists(os.path.join(OUT_DIR, f["path"]))]

print(f"Files on the CIC site: {len(files)}  |  on Colab disk: {len(all_files)}  (CSV: {len(csvs)})")
print(f"Total size: {sum(os.path.getsize(p) for p in all_files)/1024**3:.1f} GB\n")
print(f"Missing files:            {len(missing)}")
print(f"Unfinished (.part) files: {len(partial)}")
print(f"Empty files:              {len(empty)}")
print(f"Web pages saved as CSV:   {len(html_junk)}")
for p in (partial + empty + html_junk)[:20]:
    print("   ⚠️", os.path.relpath(p, OUT_DIR))

# re-download anything missing
for f in missing:
    dest = os.path.join(OUT_DIR, f["path"])
    try:
        size = download_file(session, f["url"], dest)
        print(f"✅ Downloaded {f['path']} ({human(size)})")
    except Exception as e:
        print("❌", f["path"], e)

if not (missing or partial or empty or html_junk):
    print("\n✅ Download complete!")

Files on the CIC site: 314  |  on Colab disk: 314  (CSV: 312)
Total size: 50.8 GB

Missing files:            0
Unfinished (.part) files: 0
Empty files:              0
Web pages saved as CSV:   0

✅ Download complete!


## 7. Convert CSV → Parquet
- zstd compression (54.5 GB → 11.3 GB)
- If a column turns out to contain text, the file is restarted with that column stored as text, so no values are lost
- Files with no title row (5 DoS-TCP_Flood files) get the standard flow column names
- Row counts are checked for every file, and a `conversion_manifest.csv` is saved

In [8]:
import os, time
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

SRC   = OUT_DIR        # the CSVs on Colab's disk
DEST  = PQ_LOCAL     # where the Parquet files go
CHUNK = 500_000                # rows read at a time (keeps memory low)

csvs = sorted(os.path.join(r, f) for r, _, fs in os.walk(SRC) for f in fs if f.lower().endswith(".csv"))
print(f"CSV files to convert: {len(csvs)}\n")

def looks_like_no_title(cols):
    """True if the 'column names' are actually data (e.g. IP addresses)."""
    first = [str(c) for c in cols[:4]]
    return not any(ch.isalpha() for ch in "".join(first[:3])) or (len(first) > 1 and first[1].count(".") == 3)

flow_example = next(p for p in csvs if "Flow Based" in p and not looks_like_no_title(pd.read_csv(p, nrows=0).columns))
FLOW_COLS = [c.strip() for c in pd.read_csv(flow_example, nrows=0).columns]

class NeedsText(Exception):
    """A column we stored as numbers turned out to contain text."""

def convert_one(src, tmp, text_cols):
    header = pd.read_csv(src, nrows=0).columns
    note = ""
    if looks_like_no_title(header):
        names = FLOW_COLS if ("Flow Based" in src and len(header) == len(FLOW_COLS)) else [f"col_{k}" for k in range(len(header))]
        reader = pd.read_csv(src, header=None, names=names, chunksize=CHUNK, low_memory=False)
        note = "had no title row: column names added"
    else:
        reader = pd.read_csv(src, chunksize=CHUNK, low_memory=False)

    writer, schema, rows, lost = None, None, 0, 0
    try:
        for chunk in reader:
            chunk.columns = [str(c).strip() for c in chunk.columns]
            before = chunk.notna().sum()                      # values per column BEFORE converting
            if schema is None:                                # first chunk decides the types
                fields = []
                for c in chunk.columns:
                    numeric = c not in text_cols and (pd.api.types.is_numeric_dtype(chunk[c]) or chunk[c].isna().all())
                    fields.append(pa.field(c, pa.float64() if numeric else pa.string()))
                schema = pa.schema(fields)
                writer = pq.ParquetWriter(tmp, schema, compression="zstd")
            for f in schema:
                c = f.name
                if pa.types.is_floating(f.type):
                    num = pd.to_numeric(chunk[c], errors="coerce").astype("float64")
                    if (chunk[c].notna() & num.isna()).any():     # text found in a number column
                        raise NeedsText(c)
                    chunk[c] = num
                else:
                    chunk[c] = chunk[c].astype("string")
            lost += int((before - chunk.notna().sum()).abs().sum())   # should always be 0
            writer.write_table(pa.Table.from_pandas(chunk[schema.names], schema=schema, preserve_index=False))
            rows += len(chunk)
    finally:
        if writer is not None:
            writer.close()
    if writer is None:                                        # title row but no data rows
        empty = pd.DataFrame(columns=[str(c).strip() for c in header]).astype("string")
        pq.write_table(pa.Table.from_pandas(empty, preserve_index=False), tmp)
    return rows, lost, note

manifest, start = [], time.time()
for i, src in enumerate(csvs, 1):
    rel = os.path.relpath(src, SRC)
    dest = os.path.join(DEST, rel[:-4] + ".parquet")
    if os.path.exists(dest):
        print(f"[{i}/{len(csvs)}] ✔ already converted: {rel}")
        continue
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    tmp, text_cols = dest + ".tmp", set()
    while True:
        try:
            rows, lost, note = convert_one(src, tmp, text_cols)
            break
        except NeedsText as e:                                # restart this file with that column as text
            text_cols.add(e.args[0])
            print(f"      ↻ column '{e.args[0]}' has text in it: restarting this file, storing it as text")
    os.replace(tmp, dest)

    p_rows = pq.ParquetFile(dest).metadata.num_rows
    mb_in, mb_out = os.path.getsize(src) / 1e6, os.path.getsize(dest) / 1e6
    ok = (p_rows == rows) and lost == 0
    manifest.append({"file": rel, "csv_rows": rows, "parquet_rows": p_rows, "values_lost": lost,
                     "text_columns": ", ".join(sorted(text_cols)), "csv_MB": round(mb_in, 1),
                     "parquet_MB": round(mb_out, 1), "note": note})
    print(f"[{i}/{len(csvs)}] {'✅' if ok else '❌ CHECK FAILED'} {rel}: {rows:,} rows, {mb_in:,.0f} MB → {mb_out:,.0f} MB  {note}")

# ---------- summary ----------
all_pq = sorted(os.path.join(r, f) for r, _, fs in os.walk(DEST) for f in fs if f.endswith(".parquet"))
m = pd.DataFrame(manifest)
if len(m):
    old = os.path.join(DEST, "conversion_manifest.csv")
    if os.path.exists(old):
        m = pd.concat([pd.read_csv(old), m]).drop_duplicates("file", keep="last")
    m.to_csv(old, index=False)
print("\n================ SUMMARY ================")
print(f"Parquet files on disk:   {len(all_pq)} / {len(csvs)}")
if len(m):
    print(f"Row counts all match:    {(m.csv_rows == m.parquet_rows).all()}")
    print(f"Values lost:             {int(m.values_lost.sum())}")
print(f"Size: {sum(os.path.getsize(p) for p in csvs)/1e9:,.1f} GB as CSV  →  "
      f"{sum(os.path.getsize(p) for p in all_pq)/1e9:,.1f} GB as Parquet")
print(f"Time this run: {(time.time() - start)/60:.0f} minutes")

CSV files to convert: 312

[1/312] ✅ Anomaly Detection - Flow Based features/Benign/BenignTraffic.pcap_Flow.csv: 183,630 rows, 99 MB → 25 MB  
[2/312] ✅ Anomaly Detection - Flow Based features/Benign/BenignTraffic1.pcap_Flow.csv: 84,526 rows, 46 MB → 13 MB  
[3/312] ✅ Anomaly Detection - Flow Based features/Benign/BenignTraffic2.pcap_Flow.csv: 91,279 rows, 50 MB → 14 MB  
[4/312] ✅ Anomaly Detection - Flow Based features/Benign/BenignTraffic3.pcap_Flow.csv: 38,895 rows, 21 MB → 6 MB  
[5/312] ✅ Anomaly Detection - Flow Based features/BruteForce/DictionaryBruteForce.pcap_Flow.csv: 3,619 rows, 2 MB → 0 MB  
[6/312] ✅ Anomaly Detection - Flow Based features/DDoS/DDoS ACK Fragmentation/DDoS-ACK_Fragmentation.pcap_Flow.csv: 323,886 rows, 160 MB → 24 MB  
[7/312] ✅ Anomaly Detection - Flow Based features/DDoS/DDoS ACK Fragmentation/DDoS-ACK_Fragmentation1.pcap_Flow.csv: 137,172 rows, 71 MB → 15 MB  
[8/312] ✅ Anomaly Detection - Flow Based features/DDoS/DDoS ACK Fragmentation/DDoS-ACK_Fragme

## 8. Copy the Parquet files to Drive

In [9]:
# Copy the Parquet files + the 2 README files to Drive, then flush so every upload really finishes
import subprocess
os.makedirs(PQ_DRIVE, exist_ok=True)
subprocess.run(["rsync", "-a", "--info=progress2", PQ_LOCAL + "/", PQ_DRIVE + "/"], check=True)

for p in glob.glob(f"{OUT_DIR}/**/*.txt", recursive=True):          # README files
    dest = os.path.join(PQ_DRIVE, os.path.relpath(p, OUT_DIR))
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy2(p, dest)

drive.flush_and_unmount()          # waits until Drive has received everything
drive.mount("/content/drive")
print("✅ Copied to", PQ_DRIVE)

Mounted at /content/drive
✅ Copied to /content/drive/MyDrive/IoT_Anomaly_Detection_Project/Data/CIC-IoT-DIAD-2024-parquet


## 9. Verify the copy
Compares every file on Colab's disk with Drive: file count, file size and row count.

In [10]:
import os
import pyarrow.parquet as pq

LOCAL = PQ_LOCAL
DRIVE = PQ_DRIVE

local_files = {os.path.relpath(os.path.join(r, f), LOCAL): os.path.getsize(os.path.join(r, f))
               for r, _, fs in os.walk(LOCAL) for f in fs if f.endswith(".parquet")}
drive_files = {os.path.relpath(os.path.join(r, f), DRIVE): os.path.getsize(os.path.join(r, f))
               for r, _, fs in os.walk(DRIVE) for f in fs if f.endswith(".parquet")}

missing  = [f for f in local_files if f not in drive_files]
wrong_sz = [f for f in local_files if f in drive_files and drive_files[f] != local_files[f]]

row_problems = []
for f in local_files:
    if f in drive_files and f not in wrong_sz:
        try:
            if pq.ParquetFile(os.path.join(DRIVE, f)).metadata.num_rows != pq.ParquetFile(os.path.join(LOCAL, f)).metadata.num_rows:
                row_problems.append(f)
        except Exception as e:
            row_problems.append(f"{f}  ({e})")

readmes = [f for _, _, fs in os.walk(DRIVE) for f in fs if f.lower().endswith(".txt")]

print(f"Parquet files on Colab's disk: {len(local_files)}   ({sum(local_files.values())/1e9:.2f} GB)")
print(f"Parquet files in Drive:        {len(drive_files)}   ({sum(drive_files.values())/1e9:.2f} GB)")
print(f"README files in Drive:         {len(readmes)}   (expected 2)\n")
print(f"Missing in Drive:              {len(missing)}")
print(f"Wrong size (incomplete copy):  {len(wrong_sz)}")
print(f"Row count doesn't match:       {len(row_problems)}")
for f in (missing + wrong_sz + row_problems)[:20]:
    print("   ⚠️", f)

if not (missing or wrong_sz or row_problems) and len(drive_files) == 312 and len(readmes) == 2:
    print("\n✅ Your Drive has the COMPLETE Parquet dataset. Every file, every row.")
else:
    print("\n⚠️ Not complete yet: run the copy cell again (it only uploads what's missing), then re-run this check.")

Parquet files on Colab's disk: 312   (11.29 GB)
Parquet files in Drive:        312   (11.29 GB)
README files in Drive:         2   (expected 2)

Missing in Drive:              0
Wrong size (incomplete copy):  0
Row count doesn't match:       0

✅ Your Drive has the COMPLETE Parquet dataset. Every file, every row.
